In [3]:
import pandas as pd

df = pd.read_csv(r"C:\Users\akars\OneDrive\Desktop\projects\End-to-End Supply Chain Analytics & Late Delivery Prediction\data\DataCoSupplyChainDataset_Cleaned.csv")

# Dataset grain and keys
print("Rows:", len(df))
print("Distinct orders:", df["Order_Id"].nunique())
print("Distinct order items:", df["Order_Item_Id"].nunique())
print("Distinct customers:", df["Customer_Id"].nunique())
print("Full-row duplicates:", df.duplicated().sum())

# Delivery status distribution
print(df["Delivery_Status"].value_counts(dropna=False))

# Overall delivery rates
late = (df["Delivery_Status"] == "Late delivery").sum()
cancelled = (df["Delivery_Status"] == "Shipping canceled").sum()
non_cancelled = len(df) - cancelled

print("Late rate across all records:", round(late / len(df) * 100, 2), "%")
print(
    "Late rate excluding cancellations:",
    round(late / non_cancelled * 100, 2), "%"
)

# Mode-level comparison
mode_summary = (
    df.groupby("Shipping_Mode")
      .agg(
          records=("Order_Item_Id", "size"),
          late_records=("Delivery_Status",
                        lambda s: (s == "Late delivery").sum()),
          cancelled_records=("Delivery_Status",
                             lambda s: (s == "Shipping canceled").sum()),
          avg_actual_days=("Days_for_shipping_(real)", "mean"),
          avg_scheduled_days=("Days_for_shipment_(scheduled)", "mean")
      )
)

mode_summary["late_rate_all_pct"] = (
    mode_summary["late_records"] / mode_summary["records"] * 100
)
mode_summary["late_rate_excl_cancelled_pct"] = (
    mode_summary["late_records"]
    / (mode_summary["records"] - mode_summary["cancelled_records"])
    * 100
)

print(mode_summary.round(2))

Rows: 180518
Distinct orders: 65751
Distinct order items: 180518
Distinct customers: 20651
Full-row duplicates: 0
Delivery_Status
Late delivery        98977
Advance shipping     41591
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64
Late rate across all records: 54.83 %
Late rate excluding cancellations: 57.29 %
                records  late_records  cancelled_records  avg_actual_days  \
Shipping_Mode                                                               
First Class       27814         26513               1301             2.00   
Same Day           9737          4454                444             0.48   
Second Class      35216         26987               1410             3.99   
Standard Class   107751         41023               4599             4.00   

                avg_scheduled_days  late_rate_all_pct  \
Shipping_Mode                                           
First Class                    1.0              95.32   
Same Day             